# Experiment 3 — Singleton Ablation (LR only)

Tests the hypothesis: **singleton ideas (support == 1) carry no
classifiable signal**, so removing them from golden B should barely move
the needle. Three conditions, identical logistic regression, same 60/40
document-level split, same seed (42):

- **A (control):** TF-IDF sentence features → label
- **B (full golden B):** X + C over all 8,706 ideas (exp2 result)
- **C (no singletons):** X + C over the 1,408 shared ideas (support ≥ 2)

C-space size drops 8,706 → 1,408 (6× smaller); mean B size drops 4.05 → 1.49.
If the +9.8pt lift (exp2) mostly survives, the extraction problem for Stage A
is the 1,408-idea space, not 8,706.


In [ ]:
# 1. Setup + fetch committed inputs
import os
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
os.chdir('/content/qsbc')
print('cwd:', os.getcwd())
!pip install -q pandas scikit-learn


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('results/exp1/samples_2858.csv')
df = df.drop_duplicates(subset=['sentence_id']).reset_index(drop=True)
ideas_full = pd.read_csv('results/exp1/sample_ideas.csv')
ideas_shared = pd.read_csv('results/exp3/sample_ideas_shared.csv')

grp_full = ideas_full.groupby('sentence_id')['idea_id'].apply(list).to_dict()
grp_shar = ideas_shared.groupby('sentence_id')['idea_id'].apply(list).to_dict()
df['ideas_full'] = df['sentence_id'].map(grp_full).fillna('').apply(
    lambda x: x if isinstance(x, list) else [])
df['ideas_shar'] = df['sentence_id'].map(grp_shar).fillna('').apply(
    lambda x: x if isinstance(x, list) else [])

print('samples:', len(df))
print('full B  mean size:', round(df['ideas_full'].str.len().mean(), 2),
      '| space:', ideas_full['idea_id'].nunique())
print('shared B mean size:', round(df['ideas_shar'].str.len().mean(), 2),
      '| space:', ideas_shared['idea_id'].nunique())
print('samples with empty shared-B:', int((df['ideas_shar'].str.len() == 0).sum()))


In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
docs = df['doc_key'].unique()
train_docs, test_docs = train_test_split(docs, test_size=0.4, random_state=SEED)
train = df[df['doc_key'].isin(train_docs)].reset_index(drop=True)
test  = df[df['doc_key'].isin(test_docs)].reset_index(drop=True)
print('doc overlap:', len(set(train_docs) & set(test_docs)))
print('train:', len(train), '| test:', len(test))
print('train classes:', train['label'].nunique(), '| test classes:', test['label'].nunique())


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

tf = TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2),
                     stop_words='english')
Xtr = tf.fit_transform(train['sentence'])
Xte = tf.transform(test['sentence'])
print('X dims:', Xtr.shape)

def build_C(col):
    mlb = MultiLabelBinarizer()
    mlb.fit(list(train[col]) + list(test[col]))
    return mlb.transform(list(train[col])), mlb.transform(list(test[col])), mlb.classes_.size

Cfull_tr, Cfull_te, n_full = build_C('ideas_full')
Cshar_tr, Cshar_te, n_shar = build_C('ideas_shar')
print('full C dims:', Cfull_tr.shape, '| shared C dims:', Cshar_tr.shape)


In [ ]:
def run(Xtr_f, Xte_f, name):
    lr = LogisticRegression(max_iter=1000, C=10, class_weight='balanced')
    lr.fit(Xtr_f, train['label'])
    p = lr.predict(Xte_f)
    acc = accuracy_score(test['label'], p)
    f1 = f1_score(test['label'], p, average='macro')
    print(f'{name:36s} acc={acc:.4f}  macro-F1={f1:.4f}')
    return acc, f1

accA, f1A = run(Xtr, Xte, 'A: X only')
accB, f1B = run(hstack([Xtr, Cfull_tr]), hstack([Xte, Cfull_te]),
                'B: X + C (full, %d)' % n_full)
accC, f1C = run(hstack([Xtr, Cshar_tr]), hstack([Xte, Cshar_te]),
                'C: X + C (shared, %d)' % n_shar)

print('\nDelta C - A:  acc %+.4f   macro-F1 %+.4f' % (accC - accA, f1C - f1A))
print('Delta B - A:  acc %+.4f   macro-F1 %+.4f' % (accB - accA, f1B - f1A))
print('Delta C - B:  acc %+.4f   macro-F1 %+.4f (cost of dropping singletons)'
      % (accC - accB, f1C - f1B))


In [ ]:
import pandas as pd
rows = [
    ['A: X -> Y (control)', accA, f1A],
    ['B: X + C -> Y (full golden B)', accB, f1B],
    ['C: X + C -> Y (no singletons)', accC, f1C],
]
print(pd.DataFrame(rows, columns=['Condition', 'Accuracy', 'Macro-F1']).to_string(index=False))
